# Study Agent — a spaced-repetition coach with persistent learner state

An interactive study coach built with `vidbyte-sdk`. The agent's tools give it
access to a real spaced-repetition scheduler: it pulls the cards you're due to
review, quizzes you with retrieval practice, grades your answers, and writes
the outcome back so the scheduler can pick the next due date.

The durable knowledge of *what you know and when you'll forget it* lives
behind the tools — the model provides the conversation and judgment.

Before running: `cp .env.example .env` and add your provider key.

## Setup

`VidbyteClient` (in [`vidbyte_client.py`](vidbyte_client.py)) talks to the
live Vidbyte API when `VIDBYTE_API_URL` / `VIDBYTE_API_KEY` are set, and
otherwise falls back to a local demo deck (`deck.json`) scheduled with a
compact SM-2 implementation — so this notebook runs end-to-end offline.

In [ ]:
import os

from dotenv import load_dotenv
from vidbyte import BaseAgent, tool

from vidbyte_client import VidbyteClient

load_dotenv()
client = VidbyteClient()

print("Learner state:", "live Vidbyte API" if client.live else "local demo deck (deck.json)")

## Define the tools

Plain Python functions with type hints and docstrings become callable tools —
the SDK handles schema description, formatting, and execution. Note that the
agent never "remembers" your mastery; it *reads* it. Swap the local stub for
the live Vidbyte API and the agent code does not change.

In [ ]:
@tool
def get_due_reviews(limit: int = 5) -> list[dict]:
    """Return the flashcards due for review right now, hardest first.

    Each card has card_id, concept, prompt, and the reference answer
    (for grading the learner — never reveal it before they attempt recall).
    """
    return client.due_reviews(limit=limit)


@tool
def get_learner_context(topic: str) -> dict:
    """Summarize what the learner knows about a topic: per-concept ease,
    review counts, lapses, and their single weakest concept."""
    return client.learner_context(topic)


@tool
def record_review(card_id: str, rating: int) -> dict:
    """Record the outcome of one card review and reschedule it.

    rating: 0 = forgot, 1 = recalled with difficulty, 2 = recalled well,
    3 = trivially easy. Returns the number of days until the card is due again.
    """
    return client.record_review(card_id, rating)

## Create the agent

In [ ]:
SYSTEM_PROMPT = """You are a study coach running a spaced-repetition session.

Session protocol:
1. Call get_due_reviews to fetch due cards. If none are due, say so and offer
   to review the learner's weakest concept anyway (use get_learner_context).
2. Quiz ONE card at a time. Always demand free recall first — ask the prompt
   and wait. Never reveal or hint at the answer before the learner attempts it.
3. Grade their attempt against the reference answer. Be honest: partial
   credit is "hard" (1), not "good" (2). Explain what they missed in one or
   two sentences — feedback matters most when they were confidently wrong.
4. Call record_review with your rating, tell them when the card returns,
   then move to the next card.
5. After the last card, give a one-paragraph session summary: what's solid,
   what's shaky, and the single concept to focus on next.

Tone: warm but rigorous. Effortful retrieval is the point — do not make it
feel easy."""

agent = BaseAgent(
    name="study-coach",
    system_prompt=SYSTEM_PROMPT,
    provider=os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai"),
    model_name=os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1"),
    tools=[get_due_reviews, get_learner_context, record_review],
    max_tool_rounds=8,
)

## Start the session

The agent's history persists across `run()` calls, so each cell below is one
conversational turn in the same session.

In [ ]:
reply = agent.run("Start my study session.")
print(reply.content)

## Answer the card

Edit `my_answer` with your own free-recall attempt, then run the cell. The
coach grades it, calls `record_review`, and moves to the next card. Re-run
this cell (with a new answer) for each card in the session.

In [ ]:
my_answer = "Retrieving from memory strengthens it more than re-reading does."

reply = agent.run(my_answer)
print(reply.content)

## Inspect the rescheduled deck

After a few turns, look at the scheduler state the agent has been writing to:
ease factors moved, intervals grew, and due dates are now in the future.

In [ ]:
import json
from pathlib import Path

if client.live:
    print(client.learner_context("learning science"))
else:
    deck = json.loads(Path("deck.json").read_text(encoding="utf-8"))
    for card in deck["cards"]:
        print(f'{card["concept"]:>24}  ease={card["ease"]:.2f}  reviews={card["reviews"]}  next due in {card["interval_days"]}d')

## Adapt it

- Point `VidbyteClient` at your own card store (Anki export, Notion DB).
- Add a `get_gap_map()` tool so the coach prioritizes weak concepts, not just due dates.
- Wrap the turn cells in a chat UI — the agent and tools are transport-agnostic.